# ReAct- Reason + Action
Thinks about what to do
Uses a tool if needed
Observes the result
Thinks again
Continues until it has enough information
=>
Question
   ↓
Reason
   ↓
Act (Tool Call)
   ↓
Observe Result
   ↓
Reason Again
   ↓
Act Again
   ↓
Observe
   ↓
Final Answer

# failure of agents
Failure 1: Tool Not Available
        using tool that is not available in the code.
Failure 2: Wrong Tool Selection
Failure 3; Infinte loops
         Reason
         Search
         Reason 
         Search 
         Reason
Failure 4: Graceful Failure
        instead crashing into an exception directly , goes to try and catch and returns a proper failure
Failure 5; Retry Logic

In [1]:
import sqlite3
from duckduckgo_search import DDGS

# create the data base for DB tool 

In [2]:
con = sqlite3.connect("employee.db")
 
cursor=con.cursor()

cursor.execute(""" CREATE TABLE IF NOT EXISTS employee( 
              employee_id INTEGER,
              employee_name TEXT,
              manager TEXT
              )
            """)
cursor.execute("DELETE FROM employee")
cursor.execute(""" INSERT INTO employee VALUES (99, 'vikas' ,'sam Altman')
               """)

cursor.execute(""" INSERT INTO employee VALUES (100, 'mike' ,'Satya Nadella')
               """)

con.commit()
con.close()
print("database ready")

database ready


# database tool

In [30]:
def db_query(employee_id):
    con = sqlite3.connect("employee.db")
    cursor= con.cursor()
    cursor.execute("""
                SELECT employee_name, manager 
                FROM employee WHERE employee_id=? """, (int(employee_id),)
                )
    result =  cursor.fetchone()

    con.close()

    if result:
        return  (
            f"Employee:{result[0]}, "
            f"Manager:{result[1]}"
        )
    return "Employee not found"

In [31]:
db_query(100)

'Employee:mike, Manager:Satya Nadella'

In [5]:
from ddgs import DDGS

In [6]:
def web_search(query):
    with DDGS() as ddgs:
        result =list(ddgs.text(query, max_results=5))

    print(result)
    
    if len(result)==0:
        return "NO result found"
    return result[0]["body"]
    # print(result)

In [7]:
web_search("Sam Altman current working in which  company")

[{'title': 'Sam Altman - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Sam_Altman', 'body': 'Samuel Harris Altman (born April 22, 1985) is an American entrepreneur and investor who has been the chief executive officer (CEO) of the artificial intelligence company OpenAI since 2019.'}, {'title': 'Sam Altman | Biography, OpenAI, ChatGPT... | Britannica Money', 'href': 'https://www.britannica.com/money/Sam-Altman', 'body': 'Sam Altman is CEO of OpenAI, the artificial intelligence company that developed ChatGPT.In 2011, Altman began working part-time as a partner at Y Combinator, and the next year he founded the venture fund Hydrazine Capital with his brother Max Altman.'}, {'title': 'Sam Altman: OpenAI CEO on GPT-4, ChatGPT, and the... - YouTube', 'href': 'https://www.youtube.com/watch?v=L_Guz73e6fw', 'body': 'Sam Altman is the CEO of OpenAI, the company behind GPT-4, ChatGPT, DALL-E, Codex, and many other state-of-the-art AI technologies. Please support this podca...'}, {'title': 'Op

'Samuel Harris Altman (born April 22, 1985) is an American entrepreneur and investor who has been the chief executive officer (CEO) of the artificial intelligence company OpenAI since 2019.'

In [8]:
web_search("CEO of OPENAI?")

[{'title': 'Sam Altman - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Sam_Altman', 'body': 'Samuel Harris Altman (born April 22, 1985) is an American entrepreneur and investor who has been the chief executive officer (CEO) of the artificial intelligence company OpenAI since 2019. Altman attended Stanford University for two years before he dropped out and co-founded Loopt, a smartphone geosocial networking service. Loopt was acquired by Green Dot Corporation for $43.4 million. [1] In ...'}, {'title': 'OpenAI - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Openai', 'body': "[14] In 2023 and 2024, OpenAI faced multiple lawsuits for alleged copyright infringement against authors and media companies whose work was used to train some of OpenAI's products. In November 2023, OpenAI's board removed Sam Altman as CEO, citing a lack of confidence in him, but reinstated him five days later following a reconstruction of the ..."}, {'title': 'Sam Altman | Biography, OpenAI, ChatGPT, & Mic

'Samuel Harris Altman (born April 22, 1985) is an American entrepreneur and investor who has been the chief executive officer (CEO) of the artificial intelligence company OpenAI since 2019. Altman attended Stanford University for two years before he dropped out and co-founded Loopt, a smartphone geosocial networking service. Loopt was acquired by Green Dot Corporation for $43.4 million. [1] In ...'

# graceful failure and retry logic

In [9]:
import time
def retry(func):
    def grace(*args):
        retry=3

        for attempt in range(retry):
            try:
                result=func(*args)
                return result
            except Exception as e:
                print(f"Retry {attempt +1}")
                time.sleep(1)
        
        return "Tool Failed"
    return grace

In [10]:
db_query_retry = retry(db_query)
web_search_retry=retry(web_search)

In [11]:
from langchain.tools import tool

/home/sathw/.pyenv/versions/3.11.9/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
@tool
def db_query_tool(employee_id: int)-> str:
    """ retrieve Employee information from database"""
    return db_query_retry(employee_id)

In [13]:
@tool
def web_search_tool(query:str)->str:
    """ search in the internet for inforamtion"""
    return web_search_retry(query)

In [14]:
print(web_search_tool.invoke("Sam Altman current company?"))

[{'title': 'Sam Altman - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Sam_Altman', 'body': 'Sam Altman has recently expanded his investment portfolio to include stakes in over 400 companies, valued at around $2.8 billion.'}, {'title': 'Who is Sam Altman?', 'href': 'https://www.lxahub.com/stories/who-is-sam-altman', 'body': 'In addition to investing and working with tech companies, Altman is an active philanthropist and mentor to young entrepreneurs.'}, {'title': 'Sam Altman’s Startup Portfolio: 14 Companies Backed by the', 'href': 'https://observer.com/2025/06/sam-altman-startup-investments/', 'body': 'Although OpenAI is currently valued at a staggering $300 billion, Altman has stated he holds no equity in the company and receives only a modest ...'}, {'title': 'Sam Altman invested $180 million into a company trying to delay', 'href': 'https://www.technologyreview.com/2023/03/08/1069523/sam-altman-investment-180-million-retro-biosciences-longevity-death/', 'body': 'All these comp

In [32]:
@tool
def sql_query_tool(query : str)->str:
    """ 
    Execute SQL queries on the employee database.
    useful for:
    -counting employees
    -Listing employees
    -Viewing schema
    -Database analytics
    """

    try:
        con =sqlite3.connect("employee.db")
        cursor= con.cursor()

        cursor.execute(query)
        result = cursor.fetchall()

        con.close()
        return str(result)
    except Exception as e:
        return f"SQL Error:{e}"

In [15]:
import boto3
import json
import os
from dotenv import load_dotenv


# AWS Credentials
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION")


# bedrock_runtime = boto3.client(
#     service_name="bedrock-runtime",
#     region_name=AWS_REGION,
#     aws_access_key_id=AWS_ACCESS_KEY_ID,
#     aws_secret_access_key=AWS_SECRET_ACCESS_KEY
# )

# MODEL_ID = "amazon.nova-micro-v1:0"

In [16]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model="amazon.nova-micro-v1:0",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

In [33]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[
        db_query_tool,
        sql_query_tool,
        web_search_tool
    ],
    system_prompt="""
    You are a ReAct agent.
    Use tools which are required.
    Think step-by-step before answering.
    """
)

In [34]:
def Agent(question):

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        }
    )

    return response["messages"][-1].content

In [19]:
print(Agent("who manages employee 100 and work as manager?"))

[{'title': 'Satya Nadella - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Satya_Nadella', 'body': 'Satya Narayana Nadella (born 19 August 1967) is an Indian-American business executive. He is the chairman and chief executive officer (CEO) of Microsoft, ...'}, {'title': 'Satya Nadella email to employees on first day as CEO - Microsoft Source', 'href': 'https://news.microsoft.com/source/2014/02/04/satya-nadella-email-to-employees-on-first-day-as-ceo/', 'body': '4 Feb 2014 · Satya Nadella email to employees on first day as CEO ... Today is a very humbling day for me. It reminds me of my very first day at Microsoft, 22 ...'}, {'title': 'Leadership Lessons from Satya Nadella - Chicago Booth', 'href': 'https://www.chicagobooth.edu/magazine/leadership-lessons-satya-nadella', 'body': "Microsoft CEO Satya Nadella, '97, shares the top three attributes he looks for in a leader, and why none of it matters without empathy."}, {'title': 'Microsoft CEO Satya Nadella has said that AI agents are r

In [67]:
print(Agent("who manages employee 101 and work as manager?"))

<thinking> It appears that employee 101 does not exist in the database. This means I cannot directly retrieve the manager's information for this employee ID. I need to inform the User that the employee is not found in the database.</thinking> 

It seems there was an error; employee 101 does not exist in our database. Could you please verify the employee ID or provide another one?


In [74]:
print(Agent("how many records are present in the data base employee databse"))

<thinking> To determine the number of records in the employee database, I need to query the database using the `db_query_tool`. However, the `db_query_tool` does not have a built-in method for counting the total number of records directly. I will need to ask the User for the employee ID or any specific criteria to perform a query that could lead to an estimation or exact count.</thinking>




In [75]:
print(Agent("What is Boson Analytics"))

[{'title': 'Boson Analytics', 'href': 'https://www.bosonanalytics.com/', 'body': "We're Boson, and we're here to help your business grow. Since our founding in 2021, we've guided our clients to make data driven decisions. Using our proven end-to-end data analytics and machine learning expertise we'll equip you and your organization with a plan to succeed. You can trust our experts to give you the best insights towards building your data roadmap. Join us today."}, {'title': 'Business Signals Analytics Company | Boston Analytics', 'href': 'https://bostonanalytics.com/', 'body': 'As a unique Business Signals Analytics Company, we track 250+ business signals on a daily basis for industries and companies to predict future trends.'}, {'title': 'About | Boson - Boson Analytics', 'href': 'https://www.bosonanalytics.com/about', 'body': 'Boson Analytics is a US Based niche provider of customized data analytics solutions. We offer comprehensive capabilities and deep data science knowledge necessa

/home/sathw/.pyenv/versions/3.11.9/lib/python3.11/site-packages/pydantic/_internal/_model_construction.py:287: ResourceWarning: unclosed <ssl.SSLSocket fd=88, family=2, type=1, proto=6, laddr=('172.18.194.223', 59418), raddr=('20.204.244.192', 443)>
  private_attributes = self.__dict__.get('__private_attributes__')


Based on the information retrieved from the web search tool, Boson Analytics is a company founded in 2021 that specializes in data analytics and machine learning. They aim to help businesses grow by guiding them to make data-driven decisions and provide them with insights to build their data roadmap. Here's a brief overview:

- **Company Name**: Boson Analytics
- **Founded**: 2021
- **Specialty**: Data analytics and machine learning expertise
- **Mission**: To guide businesses to make data-driven decisions and equip organizations with a plan to succeed through their insights.

If you need more specific information or have any other questions, feel free to ask!


In [76]:
print(Agent("Who is Prem Boinpally? "))

[{'title': 'Prem Boinpally - Principal | Data Science | Data Technology - LinkedIn', 'href': 'https://www.linkedin.com/in/premboinpally', 'body': 'Experience · Principal · Advisor – Product Development, Strategy and North America Sales · Investor and Advisor – Product Development and Fund Raising · Senior ...'}, {'title': 'Meghana Boinpally - Data Analytics | Empowering Data-Driven Growth', 'href': 'https://www.linkedin.com/in/meghanaboinpally', 'body': 'Data Analytics | Empowering Data-Driven Growth | DSCSA | Technical Project Management | Supply Chain Management · As an enthusiastic learner, I use data to ...'}, {'title': 'Fabulyst - AI and ML Startup, Udaipur | YNOS', 'href': 'https://www.ynos.in/startup/fabulyst-311303', 'body': '... Prem Boinpally. Invested VC Funds - 50K Ventures & 1 other ... Fabulyst is funded by Prem Boinpally. Which VC Investors & Angel Networks funded Fabulyst ...'}, {'title': 'Fabulyst AI - LinkedIn', 'href': 'https://www.linkedin.com/company/fabulyst', 'bo

In [ ]:
print(Agent("how many people are there in the database?"))
#before adding the sql_query_tool !

<thinking> The User is asking for the total number of people in the database. However, the available tools do not provide a direct way to count all records in the database. Therefore, I cannot directly answer this question using the provided tools. I need to inform the User that I cannot provide this information.</thinking>
<response>I'm sorry, but I cannot provide the total number of people in the database using the tools available to me.</response>


In [38]:
print(Agent("how many  people are there in the employee database?"))

<thinking> The query has successfully returned the count of rows in the "employee" table, which indicates the number of employees in the database. The result is 2.</thinking> <respond> There are 2 people in the employee database.</respond>


In [ ]:
print(Agent("how many employees are there?"))
# not specifying the data base?

<thinking> It seems there was an error because the table 'employees' does not exist in the database. I should inform the user about this issue and ask if there is another way to retrieve the count of employees. </thinking> 

Sorry, it appears that there is no 'employees' table in the database. Could you please specify another way to retrieve the count of employees or provide more context on how to access this information?


In [40]:
print(Agent("how many employees are present in the google?"))

[{'title': 'Alphabet: number of employees 2025| Statista', 'href': 'https://www.statista.com/statistics/273744/number-of-full-time-google-employees/', 'body': 'Statistic: Number of full-time Alphabet employees from 2007 to 2025. You need a Statista Account for unlimited access. Immediate access to 1m+ statistics.'}, {'title': 'Google | LinkedIn', 'href': 'https://www.linkedin.com/company/google', 'body': "Google | 41,845,139 followers on LinkedIn. A problem isn't truly solved until it's solved for all. Googlers build products that help create opportunities for everyone, whether down the street or across the globe. Bring your insight, imagination and a healthy disregard for the impossible."}, {'title': 'liveabout.com/google-headquarters-offices-2892790', 'href': 'https://www.liveabout.com/google-headquarters-offices-2892790', 'body': '• Number of Google employees 2018 | Statista.'}, {'title': 'Top 5 US-based Tech Companies by Number of Employees', 'href': 'https://techbehemoths.com/blog

In [42]:
print(Agent("how many departments are there in google?"))

[{'title': 'Browse a list of Google’s office locations - About Google', 'href': 'https://about.google/company-info/locations/', 'body': 'Google has offices in nearly 60 countries. View a directory of our locations around the world.'}, {'title': 'Google DeepMind Overview, Address & Contact', 'href': 'https://prospeo.io/c/google-deepmind', 'body': 'Employees by Department. Google DeepMind has 4,301 employees across 20 departments. Departments. Number of employees.'}, {'title': 'Contact us – Google', 'href': 'https://www.google.com/intl/en_in/contact/grievance-officer.html', 'body': '- You can also find details about how to contact Google Pay India support at the Help Center page here. - Customer Care number toll free at: 1-800-419-0157. (b) For reporting any other matter pertaining to Google products.'}, {'title': 'Google Account Help', 'href': 'https://support.google.com/accounts/?hl=en', 'body': 'Learn how you can improve your Google Account. 2-Step Verification. Add an extra layer of 